# Python Virtual Environments

> 📘 **Python Mastery** · Module 05 — Intermediate Python · Lesson 7/7

One machine, many projects, conflicting needs. A virtual environment gives every project its own private Python and its own private site-packages -- so upgrading one project never breaks another.

## 🎯 Learning Objectives

- Explain the dependency-conflict problem venvs solve
- Create a virtual environment with `python -m venv .venv`
- Activate it on Windows (PowerShell, cmd, Git Bash) and macOS/Linux
- Verify from *inside* Python that a venv is active (`sys.prefix` vs `sys.base_prefix`)
- Follow the full project workflow: create, activate, install, freeze, ignore `.venv`
- Select a venv interpreter in VS Code; know the modern alternatives (uv, poetry, conda)

## 1. The Dependency-Conflict Problem

Picture two projects sharing one global Python:

- **Project A** -- built on `numpy 1.x`; its model-training script breaks under `numpy 2.x`.
- **Project B** -- needs `numpy 2.x` for its new pipeline.

Install globally and every upgrade is a coin flip: fixing B silently breaks A. Multiply by five projects and ten packages each and you get "dependency hell".

A virtual environment ends the argument: each project carries its own package shelf, so A keeps numpy 1.x forever while B enjoys 2.x.

## 2. What Exactly Is a Virtual Environment?

A **virtual environment** is a self-contained folder containing:

| Piece | Purpose |
|---|---|
| `python` / `python.exe` | launcher wired to this environment |
| `Lib/site-packages/` | this project's PRIVATE packages |
| `Scripts/activate` or `bin/activate` | shell script that switches your terminal in |
| `pyvenv.cfg` | points back to the base interpreter |

Activate it, and `python` and `pip` quietly refer to the venv versions. Deactivate, and your terminal returns to normal.

> 🔍 **Under the Hood:** a venv is not a copy of Python -- it's a thin shim. The folder gets a small config file (`pyvenv.cfg`) pointing back to your real installation plus an empty `site-packages`. When Python starts inside a venv, its startup machinery redirects package lookups to that local folder while reusing the standard library from the base install. That's why creating one takes seconds, not minutes.

## 3. Creating One

From inside your project folder:

```bash
cd my_project
python -m venv .venv
```

The conventional name is `.venv` (dot-prefixed so tools hide it); `venv` also works. Seconds later you have a fresh, nearly-empty environment -- only pip itself is preinstalled.

On Windows Git Bash, if `python` isn't found try `py -m venv .venv`.

## 4. Activating It (per platform)

Activation runs a small script that adjusts your current shell's PATH. Pick the line matching YOUR shell:

| Platform / shell | Command |
|---|---|
| Windows PowerShell | `.venv\Scripts\Activate.ps1` |
| Windows CMD | `.venv\Scripts\activate.bat` |
| Windows Git Bash | `source .venv/Scripts/activate` |
| macOS / Linux (bash, zsh) | `source .venv/bin/activate` |

Success looks like this -- the prompt grows a prefix:

```text
(.venv) C:\Users\fahim\my_project>
```

Leaving is universal:

```bash
deactivate
```

PowerShell refusing to run Activate.ps1? That's the execution policy, fixable once per user:
`Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope CurrentUser`

## 5. Am I Inside a Venv? Ask Python

Terminals lie; Python doesn't. Every interpreter knows its own `sys.prefix` (where IT looks for packages) and `sys.base_prefix` (the real installation). In a venv they differ.

**Example:** run these cells anywhere -- they safely report the truth of whatever environment executes them.

In [ ]:
import sys

print("executable :", sys.executable)
print("prefix     :", sys.prefix)
print("base       :", sys.base_prefix)
inside_venv = sys.prefix != sys.base_prefix
print("in a venv? :", inside_venv)
if inside_venv:
    print("-> packages installed now land INSIDE", sys.prefix)
else:
    print("-> currently using the base/global Python")

In [ ]:
import os

print("VIRTUAL_ENV env var set:", "VIRTUAL_ENV" in os.environ)

import venv  # the module behind 'python -m venv'
print("venv builder available:", hasattr(venv, "EnvBuilder"))

## 6. The Standard Workflow (memorise this)

```bash
# 1. once per project
cd my_project
python -m venv .venv

# 2. every session
source .venv/Scripts/activate        # Windows Git Bash (.ps1/.bat on PS/cmd)

# 3. install what the project needs
python -m pip install requests pandas

# 4. snapshot the exact set
python -m pip freeze > requirements.txt

# 5. keep git clean: add '.venv/' to .gitignore
echo ".venv/" >> .gitignore
```

Teammates (or future you, or a server) rebuild everything with:

```bash
python -m venv .venv && source .venv/Scripts/activate
python -m pip install -r requirements.txt
```

Bonus: you don't even need to activate to use it -- call the venv's python directly: `.venv/Scripts/python train.py`.

## 7. VS Code: Selecting the Interpreter

VS Code doesn't guess well when several Pythons exist. Point it at your venv once:

1. Open the project folder (**File > Open Folder**) so the venv sits in the workspace.
2. Press **Ctrl+Shift+P**, type **Python: Select Interpreter**, hit Enter.
3. Choose the entry ending in `('.venv')`.

The chosen interpreter shows in the bottom-right status bar; new integrated terminals auto-activate the venv, and the Run button uses it too. Red squiggles on freshly installed imports usually just mean VS Code is still pointed at the wrong interpreter.

## 8. Modern Alternatives (one-liners each)

The built-in `venv` + `pip` pair remains the universal baseline, but you'll meet these in the wild:

- **uv** -- a Rust-powered pip/venv replacement, dramatically faster; growing fast in data/ML shops.
- **poetry** -- project manager bundling dependencies, lockfiles and publishing.
- **conda** -- environment manager popular in data science; also installs non-Python binaries (CUDA, MKL).

They all solve the same problem -- isolated, reproducible environments -- so learning venv first makes every alternative obvious.

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Running `Activate.ps1` in cmd (or `activate.bat` in PowerShell) | wrong script for the shell; odd errors | match the table in section 4 |
| Installing before activating | packages land in the GLOBAL python anyway | check the `(.venv)` prompt or run the section-5 cell |
| Moving or renaming the project folder | venv paths are baked in; it stops working | delete `.venv`, recreate it, reinstall from requirements.txt |
| Committing `.venv` to git | thousands of binary files, useless on other machines | add `.venv/` to `.gitignore`; commit requirements.txt instead |
| Expecting global packages inside a fresh venv | a new venv starts EMPTY by design | reinstall there, or (rarely) create with `--system-site-packages` |
| Creating the venv outside the project | hard to find later; confusing tooling | standard spot: `<project>/.venv` |

## 💡 Best Practices & Pro Tips

- One `.venv` per project, always named `.venv` -- tools, tutorials and teammates all recognise it.
- Treat environments as disposable: requirements.txt is the source of truth, the folder is just a cache.
- Freeze after every intentional dependency change, before pushing.
- Windows users: fix the PowerShell execution policy once, then activation is friction-free everywhere.
- **AI-engineering relevance:** reproducible experiments rest on three pillars -- frozen environment, fixed random seed, versioned data. GPU work makes pinning critical (torch/CUDA pairs must match), and Docker images typically build a venv from requirements.txt at image-build time.

## 📌 Summary

| Command / Check | What it does |
|---|---|
| `python -m venv .venv` | create an isolated environment |
| `.venv\Scripts\Activate.ps1` | activate (Windows PowerShell) |
| `.venv\Scripts\activate.bat` | activate (Windows CMD) |
| `source .venv/Scripts/activate` | activate (Windows Git Bash) |
| `source .venv/bin/activate` | activate (macOS/Linux) |
| `deactivate` | leave the environment |
| `sys.prefix != sys.base_prefix` | Python-side proof you're inside a venv |
| Ctrl+Shift+P -> Python: Select Interpreter | point VS Code at the venv |

Key takeaways:

- Conflicting versions across projects are normal; isolation is the cure.
- A venv is a cheap shim over your real Python: seconds to make, disposable by design.
- Workflow: create -> activate -> install -> freeze -> commit requirements.txt, ignore `.venv/`.
- Verify with `sys.prefix`/`sys.base_prefix`, not the terminal prompt alone.

## 🔗 Next Lesson

That completes **Module 05 — Intermediate Python**! 🎓

Up next: **[06_File_Handling](../../06_File_Handling/)** -- reading, writing and managing files like a professional.